# Pub AI - vLLM Server on Google Colab

This notebook runs the `suckunalol/pub-ai-merged` model as an OpenAI-compatible API server using vLLM, then exposes it to the internet via ngrok.

## Before you start

1. **Enable GPU**: Go to `Runtime > Change runtime type > T4 GPU` (or A100 if you have Colab Pro).
2. **Get a free ngrok auth token**: Sign up at [ngrok.com](https://dashboard.ngrok.com/signup), then copy your token from [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken).
3. **Run each cell in order** with `Shift+Enter`.

## Important notes

- **Free Colab sessions time out after ~2-4 hours** of runtime. Your server will stop when the session ends.
- **Do not close the browser tab** while the server is running -- the session may disconnect.
- **The ngrok URL changes every time** you restart. Copy the new URL after each restart.
- **GPU memory**: T4 has 16 GB VRAM. If the model is too large, the cell will error. Try A100 (Colab Pro) if that happens.

## Cell 1 - Check GPU

In [ ]:
# Verify that a GPU is available
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime > Change runtime type > T4 GPU, then restart."
    )
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"\nGPU ready: {gpu_name} ({gpu_mem:.1f} GB)")

## Cell 2 - Install vLLM and ngrok

In [ ]:
# Install vLLM (includes OpenAI-compatible server) and pyngrok
# This takes 3-5 minutes on a fresh Colab runtime.
!pip install vllm pyngrok --quiet
print("\nInstallation complete.")

## Cell 3 - Set your ngrok auth token

Paste your token from [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) between the quotes below.

In [ ]:
# Replace the empty string with your ngrok auth token
NGROK_AUTH_TOKEN = ""  # <-- Paste your token here

if not NGROK_AUTH_TOKEN:
    raise ValueError(
        "You need an ngrok auth token. "
        "Sign up free at https://ngrok.com and paste your token above."
    )

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok authenticated.")

## Cell 4 - Start vLLM server and expose via ngrok

This cell:
1. Starts vLLM serving `suckunalol/pub-ai-merged` on port 8000
2. Waits for the server to be ready
3. Creates an ngrok tunnel and prints your public URL

**The model download may take a few minutes on first run.**

In [ ]:
import subprocess, time, requests
from pyngrok import ngrok

MODEL = "suckunalol/pub-ai-merged"
PORT = 8000

# Start vLLM in the background
process = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--host", "0.0.0.0",
        "--port", str(PORT),
        "--trust-remote-code",
        "--dtype", "auto",
        "--max-model-len", "4096",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(f"Starting vLLM with {MODEL}...")
print("Waiting for server to be ready (this can take 2-5 minutes)...")

# Wait for the server to respond
for i in range(300):  # up to 5 minutes
    time.sleep(1)
    try:
        r = requests.get(f"http://localhost:{PORT}/health")
        if r.status_code == 200:
            break
    except requests.ConnectionError:
        pass
    if i % 15 == 14:
        print(f"  Still loading... ({i+1}s)")
else:
    # Print server output for debugging
    process.terminate()
    output = process.stdout.read().decode()
    print("Server failed to start. Output:")
    print(output[-3000:])
    raise RuntimeError("vLLM server did not start within 5 minutes.")

print("vLLM server is running.")

# Open ngrok tunnel
public_url = ngrok.connect(PORT, "http").public_url

print("\n" + "=" * 60)
print("  YOUR API IS LIVE")
print("=" * 60)
print(f"\n  Base URL:        {public_url}")
print(f"  Chat endpoint:   {public_url}/v1/chat/completions")
print(f"  Models endpoint: {public_url}/v1/models")
print(f"  Model name:      {MODEL}")
print("\n" + "=" * 60)
print("\nKeep this tab open. The server stops when the session ends.")
print("Free Colab sessions last ~2-4 hours.")

## Cell 5 - Test the API (optional)

Run this cell to send a test request and confirm everything works.

In [ ]:
import requests, json

response = requests.post(
    f"{public_url}/v1/chat/completions",
    headers={"Content-Type": "application/json"},
    json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "Hello, who are you?"}],
        "max_tokens": 128,
    },
)

data = response.json()
print("Status:", response.status_code)
print("Reply:", data["choices"][0]["message"]["content"])

## Cell 6 - Monitor server logs (optional)

Run this to stream the last lines from the vLLM process. Interrupt the cell (`Ctrl+M I`) to stop watching.

In [ ]:
# Stream vLLM output -- interrupt this cell to stop
import select, sys

try:
    while True:
        line = process.stdout.readline()
        if line:
            sys.stdout.write(line.decode())
        elif process.poll() is not None:
            print("\nServer process has exited.")
            break
except KeyboardInterrupt:
    print("\nStopped watching logs.")